# Chapter 10: Optimization View of Estimation

<a href="../lite/lab/index.html?path=ch10_optimization_estimation.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize_scalar, minimize

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

There are two ways to think about estimation. The Bayesian says: "I will update my belief
every time new data arrives." The optimizer says: "I will collect all the data, then find
the single best answer that explains everything." Surprisingly, for Gaussian problems,
they give **exactly the same answer**. This chapter shows why, and introduces **least
squares** as the foundation of modern SLAM.

While Chapter 9 built the Bayes filter (the recursive Bayesian approach), this chapter
develops the optimization perspective. By the end, you will see that these two views
are complementary tools for the same underlying problem.

```{admonition} What you will build
:class: tip

- Estimate a robot's position from range measurements to beacons using least squares
- Prove that Bayesian estimation and least squares give the same answer for Gaussian problems
- Compare batch optimization with sequential Kalman filtering on the same data
- Visualize the cost surface and understand why local minima are dangerous

**Real world application:** Least squares is the computational engine of graph SLAM and bundle adjustment. After this chapter, you will understand the optimization view that powers all modern SLAM back ends.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **scipy.optimize.least_squares** | General nonlinear least squares solver with robust loss functions |
| **Ceres Solver** | Google's C++ optimization library, the most widely used in robotics for least squares |
| **g2o** | Graph optimization framework used in ORB-SLAM and many other SLAM systems |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 10.1 Maximum Likelihood Estimation

The simplest optimization approach to estimation: given data $z$, find the parameter $x$
that makes the observed data **most probable**:

$$x^* = \arg\max_x \; p(z \mid x)$$

The function $p(z \mid x)$, viewed as a function of $x$ (with $z$ fixed), is called the
**likelihood function**.

**Key insight:** we are not asking "what is the probability of $x$?" We are asking
"for which $x$ is the observed data most likely?" These are different questions with
the same practical effect.

### Example: Estimating Distance to a Wall

A robot takes several range measurements to a wall. Each measurement is the true distance
plus Gaussian noise. What is the most likely wall distance?

In [ ]:
# ── PARAMETERS ── change these and re-run ────
true_distance = 4.2          # true wall distance (meters)
sigma = 0.5                  # measurement noise std
measurements = [3.9, 4.5, 4.1, 3.8, 4.3]   # observed range readings

# compute likelihood for a range of candidate distances
x_candidates = np.linspace(2, 6, 500)

# likelihood = product of individual Gaussian likelihoods
log_likelihood = np.zeros_like(x_candidates)
for z in measurements:
    log_likelihood += stats.norm.logpdf(z, loc=x_candidates, scale=sigma)

likelihood = np.exp(log_likelihood - log_likelihood.max())  # normalize for plotting

# ML estimate = sample mean (for Gaussian noise)
ml_estimate = np.mean(measurements)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# individual likelihoods
ax = axes[0]
colors_list = ['steelblue', 'tomato', 'orange', 'forestgreen', 'purple']
for i, z in enumerate(measurements):
    li = stats.norm.pdf(x_candidates, loc=z, scale=sigma)
    ax.plot(x_candidates, li, color=colors_list[i % len(colors_list)],
            alpha=0.6, label=f'$z_{i+1}={z}$')
ax.set_xlabel('wall distance $x$', fontweight='bold')
ax.set_ylabel('likelihood', fontweight='bold')
ax.set_title('Individual Measurement Likelihoods', fontweight='bold')
ax.legend(fontsize=8)

# combined likelihood
ax = axes[1]
ax.plot(x_candidates, likelihood, color='steelblue', linewidth=2.5)
ax.axvline(ml_estimate, color='tomato', linestyle='--', linewidth=2,
           label=f'ML estimate = {ml_estimate:.2f}')
ax.axvline(true_distance, color='forestgreen', linestyle=':', linewidth=2,
           label=f'true distance = {true_distance}')
ax.fill_between(x_candidates, likelihood, alpha=0.15, color='steelblue')
ax.set_xlabel('wall distance $x$', fontweight='bold')
ax.set_ylabel('likelihood (normalized)', fontweight='bold')
ax.set_title('Combined Likelihood Function', fontweight='bold')
ax.legend()

plt.suptitle('Maximum Likelihood Estimation', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Measurements:  {measurements}")
print(f"ML estimate:   {ml_estimate:.3f} m  (= sample mean for Gaussian noise)")
print(f"True distance: {true_distance} m")
print(f"\nThe ML estimate is the peak of the combined likelihood function.")

### From Maximizing Likelihood to Minimizing Cost

Since $\log$ is monotonically increasing, maximizing the likelihood is equivalent to
maximizing the **log likelihood**:

$$x^* = \arg\max_x \; \log p(z \mid x) = \arg\max_x \sum_i \log p(z_i \mid x)$$

And maximizing is the same as **minimizing the negative**:

$$x^* = \arg\min_x \; \left[-\sum_i \log p(z_i \mid x)\right]$$

This is the bridge to **optimization**. We will see next that for Gaussian noise, this
negative log likelihood becomes a familiar sum of squares.

## 10.2 Least Squares

When each measurement has **Gaussian noise** $z_i = h_i(x) + \epsilon_i$ with
$\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$, the negative log likelihood becomes:

$$-\log p(z \mid x) = \text{const} + \frac{1}{2} \sum_i \frac{(z_i - h_i(x))^2}{\sigma_i^2}$$

Dropping the constant, the ML estimate solves:

$$x^* = \arg\min_x \sum_i \frac{(z_i - h_i(x))^2}{\sigma_i^2}$$

This is **weighted least squares**. The weight $w_i = 1/\sigma_i^2$ makes precise
measurements count more.

### Example: Line Fitting

A robot observes landmarks at known x positions and measures their y coordinates with noise.
Fit a line $y = ax + b$ through the noisy data.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_points = 12              # number of landmark observations
noise_level = 0.8          # measurement noise std
true_slope = 1.5           # true line: y = 1.5x + 0.5
true_intercept = 0.5
seed = 17

rng = np.random.default_rng(seed)

# generate data
x_data = np.linspace(0, 5, n_points)
y_true = true_slope * x_data + true_intercept
y_noisy = y_true + rng.normal(0, noise_level, n_points)

# solve least squares: y = ax + b  =>  [x, 1] @ [a, b]^T = y
A = np.column_stack([x_data, np.ones(n_points)])
params_ls, residuals_sum, rank, sv = np.linalg.lstsq(A, y_noisy, rcond=None)
a_hat, b_hat = params_ls

# fitted line
x_fit = np.linspace(-0.5, 5.5, 200)
y_fit = a_hat * x_fit + b_hat
y_fit_true = true_slope * x_fit + true_intercept

# residuals
y_predicted = a_hat * x_data + b_hat
residuals = y_noisy - y_predicted

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# data + fit
ax = axes[0]
ax.scatter(x_data, y_noisy, color='steelblue', s=60, zorder=5, label='noisy observations')
ax.plot(x_fit, y_fit, color='tomato', linewidth=2, label=f'LS fit: y={a_hat:.2f}x+{b_hat:.2f}')
ax.plot(x_fit, y_fit_true, color='forestgreen', linewidth=1.5, linestyle='--',
        label=f'true: y={true_slope}x+{true_intercept}')
# draw residuals
for i in range(n_points):
    ax.plot([x_data[i], x_data[i]], [y_noisy[i], y_predicted[i]],
            color='orange', linewidth=1, alpha=0.7)
ax.set_xlabel('x', fontweight='bold')
ax.set_ylabel('y', fontweight='bold')
ax.set_title('Least Squares Line Fit', fontweight='bold')
ax.legend(fontsize=9)

# cost surface
ax = axes[1]
a_range = np.linspace(0.5, 2.5, 100)
b_range = np.linspace(-1.0, 2.0, 100)
AA, BB = np.meshgrid(a_range, b_range)
cost_surface = np.zeros_like(AA)
for i in range(n_points):
    cost_surface += (y_noisy[i] - AA * x_data[i] - BB) ** 2

cp = ax.contourf(AA, BB, cost_surface, levels=30, cmap='Blues')
ax.plot(a_hat, b_hat, 'o', color='tomato', markersize=10, label=f'LS solution ({a_hat:.2f}, {b_hat:.2f})')
ax.plot(true_slope, true_intercept, '*', color='forestgreen', markersize=12,
        label=f'true ({true_slope}, {true_intercept})')
ax.set_xlabel('slope $a$', fontweight='bold')
ax.set_ylabel('intercept $b$', fontweight='bold')
ax.set_title('Cost Surface $\\sum(z_i - ax_i - b)^2$', fontweight='bold')
ax.legend(fontsize=9)
plt.colorbar(cp, ax=ax, label='cost')

plt.suptitle('Least Squares Estimation', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"True parameters:  slope={true_slope}, intercept={true_intercept}")
print(f"LS estimate:      slope={a_hat:.3f}, intercept={b_hat:.3f}")
print(f"Sum of squared residuals: {np.sum(residuals**2):.3f}")

**Note:** The orange lines show the **residuals**, the vertical distance between each
observation and the fitted line. Least squares minimizes the sum of the squared lengths
of these orange lines. The cost surface on the right shows that the minimum (red dot)
is close to the true parameters (green star).

## 10.3 Residuals and Cost

The **residual** for measurement $i$ is the difference between what we observed and what
we expected:

$$r_i(x) = z_i - h_i(x)$$

The **cost function** (also called the **objective**) is the sum of squared weighted
residuals:

$$J(x) = \sum_i w_i \, r_i(x)^2 = \sum_i \frac{r_i(x)^2}{\sigma_i^2}$$

Residuals are the building blocks of every least squares solver. Large residuals signal
either noise, model errors, or **outliers**.

Let us visualize residuals for a nonlinear example: fitting the position of a robot
given range measurements to two known landmarks.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
true_robot_pos = np.array([3.0, 2.0])   # true robot position
landmarks = np.array([[0.0, 0.0],       # landmark positions
                       [6.0, 0.0],
                       [3.0, 5.0],
                       [1.0, 4.0]])
sigma_range = 0.4                        # range measurement noise
seed = 33

rng = np.random.default_rng(seed)

# true ranges + noise
true_ranges = np.linalg.norm(landmarks - true_robot_pos, axis=1)
measured_ranges = true_ranges + rng.normal(0, sigma_range, len(landmarks))


def cost_fn(pos, lm, z, sigma):
    """Sum of squared weighted residuals."""
    predicted = np.linalg.norm(lm - pos, axis=1)
    residuals = z - predicted
    return np.sum((residuals / sigma) ** 2)


# solve via optimization
result = minimize(lambda p: cost_fn(p, landmarks, measured_ranges, sigma_range),
                  x0=[0, 0], method='Nelder-Mead')
est_pos = result.x

# compute cost surface
gx = np.linspace(-1, 7, 150)
gy = np.linspace(-1, 6, 120)
GX, GY = np.meshgrid(gx, gy)
cost_grid = np.zeros_like(GX)
for k in range(len(landmarks)):
    dist = np.sqrt((GX - landmarks[k, 0])**2 + (GY - landmarks[k, 1])**2)
    cost_grid += ((measured_ranges[k] - dist) / sigma_range) ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# residuals visualization
ax = axes[0]
# draw measurement rays
predicted_ranges = np.linalg.norm(landmarks - est_pos, axis=1)
residuals = measured_ranges - predicted_ranges
max_abs_r = max(abs(residuals.min()), abs(residuals.max())) + 0.01

for k in range(len(landmarks)):
    direction = (landmarks[k] - est_pos)
    direction = direction / np.linalg.norm(direction)
    # draw line from robot to landmark
    ax.plot([est_pos[0], landmarks[k, 0]], [est_pos[1], landmarks[k, 1]],
            color='steelblue', linewidth=1, alpha=0.4)
    # draw measured range as a circle segment
    end_meas = est_pos + direction * measured_ranges[k]
    ax.plot([est_pos[0], end_meas[0]], [est_pos[1], end_meas[1]],
            color='orange', linewidth=2.5, alpha=0.7)
    # color residual by magnitude
    r_color = 'forestgreen' if abs(residuals[k]) < sigma_range else 'tomato'
    ax.annotate(f'r={residuals[k]:.2f}',
                xy=((est_pos[0] + landmarks[k, 0]) / 2,
                    (est_pos[1] + landmarks[k, 1]) / 2),
                fontsize=9, fontweight='bold', color=r_color,
                ha='center', va='bottom')

ax.scatter(*est_pos, color='tomato', s=120, zorder=5, marker='o', label='estimated pos')
ax.scatter(*true_robot_pos, color='forestgreen', s=120, zorder=5, marker='*', label='true pos')
ax.scatter(landmarks[:, 0], landmarks[:, 1], color='steelblue', s=100, zorder=5,
           marker='^', label='landmarks')
ax.set_xlabel('x (m)', fontweight='bold')
ax.set_ylabel('y (m)', fontweight='bold')
ax.set_title('Residuals: Measured vs Predicted Ranges', fontweight='bold')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# cost surface
ax = axes[1]
cp = ax.contourf(GX, GY, np.log10(cost_grid + 1), levels=30, cmap='Blues')
ax.plot(*est_pos, 'o', color='tomato', markersize=10, label='LS estimate')
ax.plot(*true_robot_pos, '*', color='forestgreen', markersize=14, label='true pos')
ax.scatter(landmarks[:, 0], landmarks[:, 1], color='orange', s=80, marker='^',
           zorder=5, label='landmarks')
ax.set_xlabel('x (m)', fontweight='bold')
ax.set_ylabel('y (m)', fontweight='bold')
ax.set_title('Cost Surface (log scale)', fontweight='bold')
ax.legend(fontsize=9)
ax.set_aspect('equal')
plt.colorbar(cp, ax=ax, label='log10(cost+1)')

plt.suptitle('Residuals and Cost Function', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"True position:      ({true_robot_pos[0]:.2f}, {true_robot_pos[1]:.2f})")
print(f"Estimated position: ({est_pos[0]:.2f}, {est_pos[1]:.2f})")
print(f"Residuals:          {residuals.round(3)}")
print(f"Cost at estimate:   {cost_fn(est_pos, landmarks, measured_ranges, sigma_range):.3f}")
print(f"\nGreen labels = small residuals (< sigma), red = large residuals.")

## 10.4 The Probability/Optimization Connection

Here is the deep connection that unifies Chapters 9 and 10. Consider a simple estimation
problem with:

- A **prior** belief: $x \sim \mathcal{N}(\mu_0, \sigma_0^2)$
- Several **measurements**: $z_i \sim \mathcal{N}(x, \sigma_z^2)$

**Bayesian approach** (from Chapter 5, Bayes' rule):

$$p(x \mid z) \propto p(z \mid x)\,p(x) \propto \exp\left\{-\frac{1}{2}\left[\frac{(x - \mu_0)^2}{\sigma_0^2} + \sum_i \frac{(z_i - x)^2}{\sigma_z^2}\right]\right\}$$

The posterior is Gaussian with mean = **MAP estimate** (Maximum A Posteriori).

**Optimization approach:** Minimize the negative log posterior:

$$x^* = \arg\min_x \left[\frac{(x - \mu_0)^2}{\sigma_0^2} + \sum_i \frac{(z_i - x)^2}{\sigma_z^2}\right]$$

This is just weighted least squares with the prior acting as an additional "measurement."

**Both give exactly the same answer.** Let us verify numerically.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
prior_mu = 3.0               # prior mean
prior_sigma = 2.0            # prior std
measurements = [4.5, 5.1, 4.8, 5.3, 4.7]   # sensor readings
sensor_sigma = 0.6           # sensor noise std

# ── BAYESIAN SOLUTION (sequential updates) ──
# start with prior
mu_bayes = prior_mu
var_bayes = prior_sigma ** 2

# update with each measurement (Kalman filter equations)
for z in measurements:
    K = var_bayes / (var_bayes + sensor_sigma ** 2)
    mu_bayes = mu_bayes + K * (z - mu_bayes)
    var_bayes = (1 - K) * var_bayes

sigma_bayes = np.sqrt(var_bayes)

# ── OPTIMIZATION SOLUTION (weighted least squares) ──
# cost = (x - prior_mu)^2 / prior_sigma^2 + sum (z_i - x)^2 / sensor_sigma^2
# take derivative, set to zero:
# x* = (prior_mu/prior_sigma^2 + sum(z_i)/sensor_sigma^2) / (1/prior_sigma^2 + n/sensor_sigma^2)
n = len(measurements)
info_prior = 1.0 / prior_sigma ** 2
info_sensor = 1.0 / sensor_sigma ** 2

mu_opt = (prior_mu * info_prior + sum(measurements) * info_sensor) / (info_prior + n * info_sensor)
var_opt = 1.0 / (info_prior + n * info_sensor)
sigma_opt = np.sqrt(var_opt)

# ── VISUALIZATION ──
x = np.linspace(1, 7, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bayesian view
ax = axes[0]
ax.plot(x, stats.norm.pdf(x, prior_mu, prior_sigma), color='steelblue', linewidth=2,
        linestyle='--', label=f'prior: $\\mu$={prior_mu}, $\\sigma$={prior_sigma}')
# likelihood (product of measurement Gaussians)
log_lik = sum(stats.norm.logpdf(x, loc=z, scale=sensor_sigma) for z in measurements)
lik = np.exp(log_lik - log_lik.max())
lik *= stats.norm.pdf(mu_bayes, mu_bayes, sigma_bayes) / lik.max()  # scale
ax.plot(x, lik, color='orange', linewidth=2, linestyle=':', label='likelihood')
ax.plot(x, stats.norm.pdf(x, mu_bayes, sigma_bayes), color='forestgreen', linewidth=2.5,
        label=f'posterior: $\\mu$={mu_bayes:.3f}, $\\sigma$={sigma_bayes:.3f}')
ax.axvline(mu_bayes, color='forestgreen', linestyle='--', alpha=0.5)
ax.set_xlabel('x', fontweight='bold')
ax.set_ylabel('density', fontweight='bold')
ax.set_title('Bayesian View: Prior \u00d7 Likelihood = Posterior', fontweight='bold')
ax.legend(fontsize=9)

# Optimization view
ax = axes[1]
cost_values = (x - prior_mu) ** 2 / prior_sigma ** 2
for z in measurements:
    cost_values += (z - x) ** 2 / sensor_sigma ** 2
ax.plot(x, cost_values, color='steelblue', linewidth=2.5, label='cost function')
ax.axvline(mu_opt, color='tomato', linestyle='--', linewidth=2,
           label=f'LS minimum = {mu_opt:.3f}')
ax.plot(mu_opt, (mu_opt - prior_mu) ** 2 / prior_sigma ** 2 +
        sum((z - mu_opt) ** 2 / sensor_sigma ** 2 for z in measurements),
        'o', color='tomato', markersize=10)
ax.set_xlabel('x', fontweight='bold')
ax.set_ylabel('cost $J(x)$', fontweight='bold')
ax.set_title('Optimization View: Minimize Cost', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Same Problem, Same Answer', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("═" * 55)
print(f"  Bayesian posterior mean:       {mu_bayes:.6f}")
print(f"  Optimization (WLS) solution:   {mu_opt:.6f}")
print(f"  Difference:                    {abs(mu_bayes - mu_opt):.2e}")
print("═" * 55)
print(f"  Bayesian posterior std:         {sigma_bayes:.6f}")
print(f"  WLS uncertainty (sqrt of var):  {sigma_opt:.6f}")
print("═" * 55)
print(f"\n  They match! For Gaussian problems, Bayes = least squares.")

**Why this matters for SLAM:** Modern SLAM systems formulate the problem as a giant
least squares optimization over all robot poses and landmark positions simultaneously.
The equivalence shown above guarantees that this optimization solution is the same as
what a Bayesian estimator would produce (under Gaussian assumptions).

## 10.5 Batch vs Incremental

There are two strategies for solving the estimation problem:

| Strategy | How it works | Pros | Cons |
|---|---|---|---|
| **Batch** | Collect all data, solve once | Optimal, uses all data | Must wait for all data; recomputes everything |
| **Incremental** | Update estimate as each measurement arrives | Real time; efficient | May accumulate numerical errors |

For Gaussian linear problems, the Kalman filter (incremental) and batch least squares
give **identical** results. Let us verify.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
n_measurements = 20          # total number of measurements
true_value = 5.0             # true quantity being estimated
sigma = 1.5                  # measurement noise std
prior_mu = 0.0               # prior mean (intentionally far off)
prior_sigma = 10.0           # prior std (very uncertain)
seed = 55

rng = np.random.default_rng(seed)
all_measurements = true_value + rng.normal(0, sigma, n_measurements)

# ── INCREMENTAL: Kalman filter style ──
kf_means = [prior_mu]
kf_stds = [prior_sigma]
mu_kf = prior_mu
var_kf = prior_sigma ** 2

for z in all_measurements:
    K = var_kf / (var_kf + sigma ** 2)
    mu_kf = mu_kf + K * (z - mu_kf)
    var_kf = (1 - K) * var_kf
    kf_means.append(mu_kf)
    kf_stds.append(np.sqrt(var_kf))

# ── BATCH: least squares using data up to each time step ──
batch_means = [prior_mu]
batch_stds = [prior_sigma]

for t in range(1, n_measurements + 1):
    z_batch = all_measurements[:t]
    info_prior = 1.0 / prior_sigma ** 2
    info_data = t / sigma ** 2
    var_b = 1.0 / (info_prior + info_data)
    mu_b = var_b * (prior_mu * info_prior + z_batch.sum() / sigma ** 2)
    batch_means.append(mu_b)
    batch_stds.append(np.sqrt(var_b))

kf_means = np.array(kf_means)
kf_stds = np.array(kf_stds)
batch_means = np.array(batch_means)
batch_stds = np.array(batch_stds)
steps = np.arange(n_measurements + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mean estimates
ax = axes[0]
ax.plot(steps, kf_means, 'o-', color='steelblue', linewidth=2, markersize=4,
        label='Kalman filter (incremental)')
ax.plot(steps, batch_means, 's--', color='tomato', linewidth=1.5, markersize=4,
        alpha=0.7, label='batch least squares')
ax.axhline(true_value, color='forestgreen', linestyle=':', linewidth=2, label='true value')
ax.fill_between(steps, kf_means - 2 * kf_stds, kf_means + 2 * kf_stds,
                alpha=0.1, color='steelblue')
ax.set_xlabel('number of measurements', fontweight='bold')
ax.set_ylabel('estimate', fontweight='bold')
ax.set_title('Mean Estimate Over Time', fontweight='bold')
ax.legend(fontsize=9)

# uncertainty
ax = axes[1]
ax.plot(steps, kf_stds, 'o-', color='steelblue', linewidth=2, markersize=4,
        label='Kalman filter (incremental)')
ax.plot(steps, batch_stds, 's--', color='tomato', linewidth=1.5, markersize=4,
        alpha=0.7, label='batch least squares')
ax.set_xlabel('number of measurements', fontweight='bold')
ax.set_ylabel('standard deviation', fontweight='bold')
ax.set_title('Uncertainty Over Time', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Batch vs Incremental: Same Answer, Different Path', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

max_diff = np.max(np.abs(kf_means - batch_means))
print(f"Maximum difference between batch and incremental means: {max_diff:.2e}")
print(f"Maximum difference between batch and incremental stds:  {np.max(np.abs(kf_stds - batch_stds)):.2e}")
print(f"\nThey are identical (up to floating point precision).")
print(f"\nFinal estimate:  {kf_means[-1]:.3f} +/- {kf_stds[-1]:.3f}")
print(f"True value:      {true_value}")

**When to use which?**

- **Incremental** (Kalman filter): real time applications where data streams in continuously
- **Batch** (least squares): offline problems, or when you want to re linearize (nonlinear
  least squares, which we will see in later SLAM chapters)

Modern SLAM systems often combine both: run an incremental filter in real time, then
periodically solve a batch optimization to refine the solution.

## 10.6 Capstone: Robot Localization from Beacon Ranges

A robot in a 2D plane takes range measurements to 5 known beacon positions. Each
measurement has different noise. We formulate and solve this as a **weighted least
squares** problem, complete with the Jacobian, normal equations, and a confidence
ellipse.

### Problem setup

Given beacons at positions $\mathbf{b}_k$ and range measurements $z_k$ with noise
$\sigma_k$, the measurement model is:

$$h_k(\mathbf{x}) = \|\mathbf{x} - \mathbf{b}_k\|$$

The Jacobian of $h_k$ with respect to $\mathbf{x} = [x, y]^T$ is:

$$\mathbf{H}_k = \frac{\partial h_k}{\partial \mathbf{x}} = \frac{(\mathbf{x} - \mathbf{b}_k)^T}{\|\mathbf{x} - \mathbf{b}_k\|}$$

The weighted least squares solution via the **normal equations**:

$$\Delta\mathbf{x} = (\mathbf{H}^T \mathbf{W} \mathbf{H})^{-1} \mathbf{H}^T \mathbf{W} \mathbf{r}$$

where $\mathbf{W} = \text{diag}(1/\sigma_k^2)$ and $\mathbf{r}$ is the residual vector.
We iterate this (Gauss Newton) until convergence.

In [ ]:
# ── PARAMETERS ── change these and re-run ────
true_pos = np.array([4.0, 3.0])    # true robot position
beacons = np.array([[0.0, 0.0],    # beacon positions
                     [8.0, 0.0],
                     [8.0, 6.0],
                     [0.0, 6.0],
                     [4.0, 7.0]])
sigmas = np.array([0.3, 0.5, 0.4, 0.6, 0.35])   # different noise per beacon
n_measurements_per_beacon = 4       # repeated measurements to each beacon
initial_guess = np.array([1.0, 1.0])  # initial guess for optimization
max_iterations = 20
seed = 77

rng = np.random.default_rng(seed)

# generate measurements (multiple per beacon)
all_z = []
all_sigma = []
all_beacon_idx = []
for k in range(len(beacons)):
    true_range = np.linalg.norm(true_pos - beacons[k])
    for _ in range(n_measurements_per_beacon):
        z = true_range + rng.normal(0, sigmas[k])
        all_z.append(z)
        all_sigma.append(sigmas[k])
        all_beacon_idx.append(k)

all_z = np.array(all_z)
all_sigma = np.array(all_sigma)
all_beacon_idx = np.array(all_beacon_idx)
n_meas = len(all_z)


def gauss_newton_step(x_est, beacons_arr, z_arr, sigma_arr, beacon_idx):
    """One Gauss Newton iteration."""
    n = len(z_arr)
    H = np.zeros((n, 2))
    r = np.zeros(n)
    W = np.diag(1.0 / sigma_arr ** 2)

    for i in range(n):
        bk = beacons_arr[beacon_idx[i]]
        diff = x_est - bk
        dist = np.linalg.norm(diff)
        if dist < 1e-10:
            dist = 1e-10
        H[i, :] = diff / dist
        r[i] = z_arr[i] - dist

    # normal equations
    HTWH = H.T @ W @ H
    HTWr = H.T @ W @ r
    dx = np.linalg.solve(HTWH, HTWr)
    cost = r.T @ W @ r
    return dx, cost, HTWH


# run Gauss Newton
x_est = initial_guess.copy()
trajectory = [x_est.copy()]
costs = []

for iteration in range(max_iterations):
    dx, cost, info_matrix = gauss_newton_step(x_est, beacons, all_z, all_sigma, all_beacon_idx)
    costs.append(cost)
    x_est = x_est + dx
    trajectory.append(x_est.copy())
    if np.linalg.norm(dx) < 1e-8:
        break

trajectory = np.array(trajectory)
n_iters_done = len(costs)

# covariance from information matrix
cov_matrix = np.linalg.inv(info_matrix)

# confidence ellipse (2 sigma)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
theta = np.linspace(0, 2 * np.pi, 200)
scale = 2.0  # 2 sigma
ellipse_x = scale * np.sqrt(eigenvalues[0]) * np.cos(theta)
ellipse_y = scale * np.sqrt(eigenvalues[1]) * np.sin(theta)
R = eigenvectors
ellipse_pts = R @ np.array([ellipse_x, ellipse_y])
ellipse_pts[0] += x_est[0]
ellipse_pts[1] += x_est[1]

# ── VISUALIZATION ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# main plot: beacons, estimate, ellipse, rays
ax = axes[0]

# draw measurement rays (one per beacon, using mean measured range)
for k in range(len(beacons)):
    direction = beacons[k] - x_est
    direction = direction / np.linalg.norm(direction)
    ax.plot([x_est[0], beacons[k, 0]], [x_est[1], beacons[k, 1]],
            color='orange', linewidth=1.0, alpha=0.5)
    # draw a circle at measured range
    mean_z_k = all_z[all_beacon_idx == k].mean()
    circ = plt.Circle(beacons[k], mean_z_k, fill=False, color='steelblue',
                      linewidth=0.8, alpha=0.3, linestyle='--')
    ax.add_patch(circ)

# confidence ellipse
ax.plot(ellipse_pts[0], ellipse_pts[1], color='tomato', linewidth=2, label='$2\\sigma$ confidence')
ax.fill(ellipse_pts[0], ellipse_pts[1], color='tomato', alpha=0.1)

# beacons
for k in range(len(beacons)):
    ax.scatter(*beacons[k], color='steelblue', s=100, marker='^', zorder=5)
    ax.annotate(f'B{k+1} ($\\sigma$={sigmas[k]})', beacons[k] + np.array([0.15, 0.15]),
                fontsize=8, color='steelblue')

# positions
ax.plot(*true_pos, '*', color='forestgreen', markersize=16, zorder=6, label='true position')
ax.plot(*x_est, 'o', color='tomato', markersize=10, zorder=6, label='LS estimate')

# optimization path
ax.plot(trajectory[:, 0], trajectory[:, 1], '.:',  color='gray', linewidth=1,
        markersize=4, alpha=0.6, label='GN iterations')

ax.set_xlabel('x (m)', fontweight='bold')
ax.set_ylabel('y (m)', fontweight='bold')
ax.set_title('Beacon Localization (Weighted Least Squares)', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.set_aspect('equal')
ax.set_xlim(-1.5, 10)
ax.set_ylim(-1.5, 9)

# convergence plot
ax = axes[1]
ax.semilogy(range(1, n_iters_done + 1), costs, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.set_xlabel('Gauss Newton iteration', fontweight='bold')
ax.set_ylabel('cost (log scale)', fontweight='bold')
ax.set_title('Convergence of Gauss Newton', fontweight='bold')

plt.suptitle('Capstone: Robot Localization from Beacon Ranges', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"True position:      ({true_pos[0]:.2f}, {true_pos[1]:.2f})")
print(f"Estimated position: ({x_est[0]:.4f}, {x_est[1]:.4f})")
print(f"Position error:     {np.linalg.norm(x_est - true_pos):.4f} m")
print(f"Converged in {n_iters_done} iterations")
print(f"\nCovariance matrix:")
print(f"  [[{cov_matrix[0,0]:.6f}, {cov_matrix[0,1]:.6f}],")
print(f"   [{cov_matrix[1,0]:.6f}, {cov_matrix[1,1]:.6f}]]")
print(f"\nPosition std (x): {np.sqrt(cov_matrix[0,0]):.4f} m")
print(f"Position std (y): {np.sqrt(cov_matrix[1,1]):.4f} m")
print(f"\n{n_meas} total measurements ({n_measurements_per_beacon} per beacon, {len(beacons)} beacons)")

### Anatomy of the Solution

Let us inspect the Jacobian and weight matrix that made this solution work.

In [ ]:
# build final Jacobian and weight matrix for inspection
H_final = np.zeros((n_meas, 2))
r_final = np.zeros(n_meas)

for i in range(n_meas):
    bk = beacons[all_beacon_idx[i]]
    diff = x_est - bk
    dist = np.linalg.norm(diff)
    H_final[i, :] = diff / dist
    r_final[i] = all_z[i] - dist

W_final = np.diag(1.0 / all_sigma ** 2)
info = H_final.T @ W_final @ H_final

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Jacobian
ax = axes[0]
im = ax.imshow(H_final, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('state dimension (x, y)', fontweight='bold')
ax.set_ylabel('measurement index', fontweight='bold')
ax.set_title('Jacobian H', fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['x', 'y'])
plt.colorbar(im, ax=ax)

# residuals
ax = axes[1]
colors_r = ['tomato' if abs(r) > 2 * all_sigma[i] else 'steelblue' for i, r in enumerate(r_final)]
ax.bar(range(n_meas), r_final, color=colors_r, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('measurement index', fontweight='bold')
ax.set_ylabel('residual (m)', fontweight='bold')
ax.set_title('Final Residuals', fontweight='bold')

# information matrix
ax = axes[2]
im2 = ax.imshow(info, cmap='Blues')
ax.set_title('Information Matrix $H^T W H$', fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['x', 'y'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['x', 'y'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{info[i,j]:.1f}', ha='center', va='center',
                fontweight='bold', fontsize=12, color='white' if info[i,j] > info.max()/2 else 'black')
plt.colorbar(im2, ax=ax)

plt.suptitle('Inside the Least Squares Solution', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print(f"The Jacobian H has shape {H_final.shape}: {n_meas} measurements, 2 state variables.")
print(f"Each row of H points from the robot toward the corresponding beacon.")
print(f"The information matrix is the 'compressed' version: how much total info we have about (x, y).")
print(f"Its inverse is the covariance matrix, which defines the confidence ellipse.")

## Exercises

**Exercise 10.1: Effect of Beacon Geometry**

Place all 5 beacons in a line (e.g., at y=0, spaced along x). Solve the localization
problem and observe the confidence ellipse. It should be very elongated in the y direction
because the beacons provide no angular diversity. Then spread the beacons around the
robot and compare. How does beacon geometry affect the shape and size of the
confidence ellipse?

**Exercise 10.2: Outlier Rejection**

Add one grossly wrong measurement (e.g., 50 meters instead of 5 meters). Observe how
the least squares solution is pulled toward the outlier. Implement a simple outlier
rejection scheme: after solving, check each residual against a threshold (e.g., $3\sigma$).
Remove measurements with residuals beyond the threshold and re solve. How many
iterations of reject and re solve does it take to recover the correct estimate?

**Exercise 10.3: Weighted vs Unweighted**

Solve the beacon localization problem twice: once with proper weights $W = \text{diag}(1/\sigma_k^2)$
and once with uniform weights (ignoring the noise differences). Compare the confidence
ellipses. When does weighting matter most? (Hint: try making one beacon very noisy
and the others very precise.)

**Exercise 10.4: Nonlinear Cost Surface**

With only 2 beacons, the cost surface has a valley (not a unique minimum). With 3 beacons,
it typically has a unique minimum. Visualize the cost surface as a 2D contour plot for
2, 3, and 5 beacons. Explain how additional beacons constrain the solution.

**Exercise 10.5: From Estimation to SLAM**

Extend the capstone: the robot does not know the beacon positions. Instead, it visits
3 known positions and takes range measurements to 5 unknown beacons from each position.
Set up the joint least squares problem: estimate both the beacon positions $[b_k^x, b_k^y]$
and an unknown offset in the robot's position. Build the full Jacobian (it should have
$2 \times 5 = 10$ unknowns for beacon positions). Solve and visualize. This is a
mini SLAM problem!

## Summary

This chapter showed that estimation can be viewed as **optimization**:

| Concept | Key Equation | Intuition |
|---|---|---|
| **Maximum Likelihood** | $x^* = \arg\max_x \; p(z \mid x)$ | Find $x$ that best explains the data |
| **Least Squares** | $x^* = \arg\min_x \sum \frac{r_i^2}{\sigma_i^2}$ | Minimize squared residuals |
| **Residual** | $r_i = z_i - h_i(x)$ | Mismatch between observed and predicted |
| **Normal Equations** | $(H^T W H)^{-1} H^T W r$ | Closed form least squares solution |
| **Bayes = Optimization** | For Gaussians, MAP = WLS | Same answer, two perspectives |
| **Batch vs Incremental** | Both converge to same answer | Trade off: accuracy vs real time |

The optimization viewpoint becomes essential in later chapters when we tackle full
**graph based SLAM**: estimating the entire robot trajectory and map simultaneously
by solving a massive (but sparse) least squares problem.